In [1]:
# Load the Drive helper and mount
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
%%capture
!pip install datasets transformers evaluate rouge_score accelerate, relplot
!pip install git+https://github.com/google-research/bleurt.git
!pip install --upgrade bitsandbytes # numpy pandas
#!pip install unbabel-comet


In [3]:
!pip -q install relplot evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [4]:
!rm -rf colab_llm_utils

In [5]:
!git clone -b multiaxial https://github.com/ravy101/colab_llm_utils.git

Cloning into 'colab_llm_utils'...
remote: Enumerating objects: 1561, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 1561 (delta 49), reused 55 (delta 24), pack-reused 1478 (from 2)
Receiving objects: 100% (1561/1561), 1.05 MiB | 5.88 MiB/s, done.
Resolving deltas: 100% (994/994), done.


In [6]:
import colab_llm_utils
from colab_llm_utils import configs

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import random
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification, AutoConfig, BitsAndBytesConfig, GenerationConfig
from accelerate import infer_auto_device_map, init_empty_weights

In [8]:

import time
import re
import sys
import os
import string
import math

In [9]:
import types
import numpy.core as np_core
import numpy as np

np._core = types.ModuleType("_core")
np._core.__dict__.update(np_core.__dict__)

In [10]:

import torch
from torch.nn import functional as F
import evaluate

In [11]:
dataset = colab_llm_utils.configs.datasets.routerbench_10k

model_config = colab_llm_utils.configs.models.qwen3_8b
large_model_config = colab_llm_utils.configs.models.qwen3_8b

#embedding_model_config = colab_llm_utils.configs.models.t5_base
short_name = model_config['model_name'].split('/')[-1]
large_model_short_name = large_model_config['model_name'].split('/')[-1]


special_tag = ''
special_tag_large = 'rag'

drive_path = f"/content/drive/MyDrive/phase3/Llama/{dataset['clean_name']}/"

FOLLOW_UP = False
PTRUE = False
SEMDEC = False

In [12]:
DICT_ANS = dataset['dict_ans']
SELF_CONF = False
SKIP_PROCESSING_SMALL = True
SKIP_PROCESSING_LARGE = True
INDEX_JOIN = True
SKIP_MERGING = True


In [13]:
LARGE_THINKING = False
if LARGE_THINKING:
  special_tag_large = "thinking"

In [14]:
K_SHOT = False
k = 5
if K_SHOT:
  special_tag = "k_shot"

In [15]:

if PTRUE:
  short_name = short_name + "ptrue"
  special_tag = "p_true"


if SELF_CONF:
  short_name = short_name + "confconf"
  special_tag = "self_conf"


if SEMDEC:
  short_name = short_name + "_semlexdec"

In [16]:
metric_dict = {}
if dataset['task_type'] == 'translation':
  metric_dict['meteor'] = colab_llm_utils.scorers.get_meteor()
  metric_dict['bleurt'] = colab_llm_utils.scorers.get_bleurt()
  metric_dict['rouge'] = colab_llm_utils.scorers.get_rouge()
  #comet = colab_llm_utils.scorers.get_comet()
elif dataset['task_type'] == 'summarization':
  metric_dict['rouge'] = colab_llm_utils.scorers.get_rouge()

process_func = colab_llm_utils.data_processing.process_dataframe

# Load Intermediate Data Files

In [17]:
files = os.listdir(drive_path)
files.sort(key=lambda x: x) #name sort
files.reverse()

In [18]:
files.reverse()

In [19]:

small_results_7 = []
results_13 = []
results_70 = []


In [ ]:
file_limit = 100
special_tag = ""
if not SKIP_PROCESSING_SMALL:
  results_7 = []
  for f in files:
    if "small" in f:
      continue
    if str(dataset["dataset_name"]) + "_" not in f:
      continue
    #print(f)
    if len(results_7) >= file_limit:
      break
    if f.endswith(".pickle") and dataset["subset"] in f and f"{short_name}{special_tag}" + "_"  in f:
      print(f"**********************************reading {f}")
      start_time = time.perf_counter()
      df = pd.read_pickle(os.path.join(drive_path, f))

      df = process_func(df, dataset, metric_dict, self_conf = SELF_CONF, p_true = PTRUE, thinking = False)
      results_7.append(df)
      end_time = time.perf_counter()

      elapsed_time = end_time - start_time
      print(f"Execution time: {elapsed_time:.4f} seconds")

  res_7 = colab_llm_utils.data_processing.combine_dataframe(results_7)
  if FOLLOW_UP:
    colab_llm_utils.data_processing.columnize_meta_field(res_7, 'follow_up')
    res_7['def_axis'] = [colab_llm_utils.data_processing.coerce_to_bounded_int(out) for out in res_7["follow_up-text"]]
  res_7.to_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name}{special_tag}.pickle"))
else:
  res_7 = pd.read_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name}{special_tag}.pickle"))

In [ ]:
large_tag = large_model_config['model_name'].split('/')[-1] + special_tag_large
if not SKIP_PROCESSING_LARGE:
  results_13 = []
  for f in files:
    if "small" in f:
      continue
    if str(dataset["dataset_name"]) + "_" not in f:
      continue
    if len(results_13) >= file_limit:
      break
    if f.endswith(".pickle") and dataset["subset"] in f and f"{large_model_short_name}{special_tag_large}" + "_"  in f:
      print(f"**********************************reading {f}")
      start_time = time.perf_counter()
      df = pd.read_pickle(os.path.join(drive_path, f))

      df = process_func(df, dataset, metric_dict, self_conf = SELF_CONF, p_true = PTRUE, thinking = False)
      results_13.append(df)
      end_time = time.perf_counter()

      elapsed_time = end_time - start_time
      print(f"Execution time: {elapsed_time:.4f} seconds")
  res_13 = colab_llm_utils.data_processing.combine_dataframe(results_13)
  res_13.to_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{large_tag}.pickle"))
else:
  try:
    res_13 = pd.read_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{large_tag}.pickle"))
  except:
    print("adding empty large.")
    res_13 = res_7[res_7['all_probas'] == False]

In [ ]:
pd.Series([len(p) for p in res_13.prompts]).mean()

np.float64(2175.311612903226)

In [ ]:
res_13.is_correct.mean()

np.float64(0.7219354838709677)

In [ ]:
if not SKIP_MERGING:
  if INDEX_JOIN:
    try:
      res_13 = res_13.drop(["ans"], axis=1)
    except:
      pass
    full_res = pd.merge(res_7, res_13, how='left', left_index=True, right_index=True, suffixes=(None,'_large'))
  else:
    full_res = pd.merge(res_7, res_13.drop(["ans"], axis=1), how='left', left_on='prompts', right_on='prompts', suffixes=(None,'_large'))
  #full_res['bleurt_13b'] = [b[0] for b in full_res['bleurt_13b']]
  if SELF_CONF:
    full_res['self_conf'] = [int(c.split("Confidence: ")[-1].strip()[:-1])/100 for c in full_res['self_conf']]
  full_res.to_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name}{special_tag}_full.pickle"))
#else:
#  full_res = pd.read_pickle(os.path.join(drive_path, f"{dataset['dataset_name']}{short_name}{special_tag}_full.pickle"))

In [ ]:
from datetime import datetime
print(f"all done {datetime.now().strftime("%H:%M:%S")}")

all done 00:28:54


In [ ]:
import pandas as pd

def summarize_accuracy_by_task(df):
    """
    Computes accuracy, sample count, and total correct responses
    per task family from the evaluated DataFrame.
    """
    # Ensure task_family is present (fallback to meta or unknown if missing)
    if 'task_family' not in df.columns and 'meta' in df.columns:
        df['task_family'] = [m.get('task_family', 'unknown') if isinstance(m, dict) else 'unknown' for m in df['meta']]

    # Group and aggregate 0/1 scores
    summary = df.groupby('task_family').agg(
        total_samples=('is_correct', 'count'),
        correct_count=('is_correct', 'sum'),
        accuracy=('is_correct', 'mean')
    ).reset_index()

    # Format accuracy as a percentage string for clean display
    summary['accuracy_pct'] = (summary['accuracy'] * 100).round(2).astype(str) + '%'

    # Sort by total sample size descending
    summary = summary.sort_values(by='total_samples', ascending=False)

    print("=== Accuracy & Breakdown by Task Family ===")
    print(summary.to_string(index=False))

    # Print overall workload accuracy
    overall_acc = df['is_correct'].mean() * 100
    print(f"\nOverall Workload Accuracy (y=1): {overall_acc:.2f}% ({df['is_correct'].sum()}/{len(df)})")

    return summary

# --- Run on your merged/processed dataframe ---
task_summary = summarize_accuracy_by_task(res_7[000:1550])

=== Accuracy & Breakdown by Task Family ===
      task_family  total_samples  correct_count  accuracy accuracy_pct
             mmlu            603            402  0.666667       66.67%
        hellaswag            464            306  0.659483       65.95%
grade-school-math            348            164  0.471264       47.13%
    arc-challenge             70             65  0.928571       92.86%
       winogrande             54             33  0.611111       61.11%
             mbpp             11              0  0.000000         0.0%

Overall Workload Accuracy (y=1): 62.58% (970/1550)


In [ ]:
task_summary = summarize_accuracy_by_task(res_7)



=== Accuracy & Breakdown by Task Family ===
      task_family  total_samples  correct_count  accuracy accuracy_pct
             mmlu           1542           1023  0.663424       66.34%
        hellaswag           1187            820  0.690817       69.08%
grade-school-math            917            668  0.728462       72.85%
    arc-challenge            172            154  0.895349       89.53%
       winogrande            146             97  0.664384       66.44%
             mbpp             36              0  0.000000         0.0%

Overall Workload Accuracy (y=1): 69.05% (2762/4000)


In [ ]:
task_summary = summarize_accuracy_by_task(new_rag)


NameError: name 'new_rag' is not defined

In [ ]:
task_summary = summarize_accuracy_by_task(res_7[new_rag.prompts == res_7.prompts])


In [ ]:
task_summary = summarize_accuracy_by_task(new_rag[new_rag.prompts == res_7.prompts])


=== Accuracy & Breakdown by Task Family ===
      task_family  total_samples  correct_count  accuracy accuracy_pct
             mmlu             45             30  0.666667       66.67%
        hellaswag             32             14  0.437500       43.75%
grade-school-math             14              3  0.214286       21.43%
    arc-challenge              5              4  0.800000        80.0%
       winogrande              3              1  0.333333       33.33%
             mbpp              1              0  0.000000         0.0%

Overall Workload Accuracy (y=1): 52.00% (52/100)


=== Accuracy & Breakdown by Task Family ===
      task_family  total_samples  correct_count  accuracy accuracy_pct
             mmlu            209            133  0.636364       63.64%
        hellaswag            142             89  0.626761       62.68%
grade-school-math            102             44  0.431373       43.14%
    arc-challenge             21             19  0.904762       90.48%
       winogrande             21             14  0.666667       66.67%
             mbpp              5              0  0.000000         0.0%

Overall Workload Accuracy (y=1): 59.80% (299/500)

In [ ]:
task_summary = summarize_accuracy_by_task(res_13[0:1550])

## K  SHOT SET

=== Accuracy & Breakdown by Task Family ===
      task_family  total_samples  correct_count  accuracy accuracy_pct

             mmlu            777            520  0.669241       66.92%

        hellaswag            598            456  0.762542       76.25%

grade-school-math            443            375  0.846501       84.65%

    arc-challenge             94             83  0.882979        88.3%

       winogrande             74             52  0.702703       70.27%
       
             mbpp             14              1  0.071429        7.14%


Overall Workload Accuracy (y=1): 74.35% (1487/2000)

## RAG SET
=== Accuracy & Breakdown by Task Family ===

      task_family  total_samples  correct_count  accuracy accuracy_pct

             mmlu            603            400  0.663350       66.33%

        hellaswag            464            332  0.715517       71.55%

  grade-school-math            348            293  0.841954        84.2%

    arc-challenge             70             59  0.842857       84.29%

       winogrande             54             34  0.629630       62.96%

             mbpp             11              1  0.090909        9.09%

Overall Workload Accuracy (y=1): 72.19% (1119/1550)

In [ ]:
(res_13['prompts'] == res_7['prompts'][:len(res_13)]).mean()

In [ ]:
pd.Series([len(r[0].split()) for r in res_13['responses']]).describe()

In [ ]:
pd.Series([len(r[0].split()) for r in res_7['responses']]).describe()

In [ ]:
res_13[res_13['prompts'].str.contains("Score")].is_correct.mean()

In [ ]:
res_7[:len(res_13)][res_13['prompts'].str.contains("Score")].is_correct.mean()

=== Accuracy & Breakdown by Task Family ===
      task_family  total_samples  correct_count  accuracy accuracy_pct
             mmlu            204            109  0.534314       53.43%
        hellaswag            175             55  0.314286       31.43%
grade-school-math            128             85  0.664062       66.41%
    arc-challenge             20             16  0.800000        80.0%
       winogrande             20             13  0.650000        65.0%
             mbpp              3              0  0.000000         0.0%

Overall Workload Accuracy (y=1): 50.55% (278/550)

BASE:
=== Accuracy & Breakdown by Task Family ===
      task_family  total_samples  correct_count  accuracy accuracy_pct
             mmlu            179            106  0.592179       59.22%
        hellaswag            159             35  0.220126       22.01%
grade-school-math            120             58  0.483333       48.33%
       winogrande             20             10  0.500000        50.0%
    arc-challenge             19             15  0.789474       78.95%
             mbpp              3              0  0.000000         0.0%

             Overall Workload Accuracy (y=1): 44.80% (224/500)

In [ ]:
colab_llm_utils.configs.datasets.routerbench_rag

In [ ]:
res_13['em'].mean()

In [ ]:
write_file = True
merge_configs = [
    #{"dataset": colab_llm_utils.configs.datasets.mixeval_ff, "model": configs.models.qwen3_8b, "special_tag": ""},
    #{"dataset": colab_llm_utils.configs.datasets.mixeval_mc, "model": configs.models.qwen3_8b, "special_tag": "k_shot"},
    #{"dataset": colab_llm_utils.configs.datasets.mixeval_ff, "model": configs.models.qwen3_8b, "special_tag": "verify"},
    #{"dataset": colab_llm_utils.configs.datasets.mixeval_ff, "model": configs.models.qwen3_8b, "special_tag": "rag"}
    #{"dataset": colab_llm_utils.configs.datasets.mixeval_ff, "model": configs.models.qwen3_4b, "special_tag": ""},
    #{"dataset": colab_llm_utils.configs.datasets.mixeval_mc, "model": configs.models.qwen3_4b, "special_tag": ""},
    # ROUTERBENCH 10K
    {"dataset": colab_llm_utils.configs.datasets.routerbench_10k, "model": configs.models.qwen3_8b, "special_tag": ""},
    {"dataset": colab_llm_utils.configs.datasets.routerbench_10k, "model": configs.models.qwen3_8b, "special_tag": "rag"},
    {"dataset": colab_llm_utils.configs.datasets.routerbench_10k, "model": configs.models.qwen3_8b, "special_tag": "k_shot"},
    {"dataset": colab_llm_utils.configs.datasets.routerbench_10k, "model": configs.models.qwen3_8b, "special_tag": "verify"},
    {"dataset": colab_llm_utils.configs.datasets.routerbench_10k, "model": configs.models.llama33_70b_chat, "special_tag": ""},
    # ROUTERBENCH KNOWLEDGE
    #{"dataset": colab_llm_utils.configs.datasets.routerbench_rag, "model": configs.models.qwen3_8b, "special_tag": ""},
    #{"dataset": colab_llm_utils.configs.datasets.routerbench_rag, "model": configs.models.qwen3_8b, "special_tag": "rag"},
    #{"dataset": colab_llm_utils.configs.datasets.routerbench_rag, "model": configs.models.qwen3_8b, "special_tag": "k_shot"},
    #{"dataset": colab_llm_utils.configs.datasets.routerbench_rag, "model": configs.models.qwen3_8b, "special_tag": "verify"},
    #{"dataset": colab_llm_utils.configs.datasets.routerbench_rag, "model": configs.models.llama33_70b_chat, "special_tag": ""},
]
result_dfs = []



for cf in merge_configs:
  drive_path = f"/content/drive/MyDrive/phase3/Llama/{cf['dataset']['clean_name']}/"
  files = os.listdir(drive_path)
  files.sort(key=lambda x: x) #name sort
  files.reverse()
  files.reverse()
  file_limit = 100
  short_name = cf['model']['model_name'].split('/')[-1]

  results_loop = []
  for f in files:
    if str(cf['dataset']["dataset_name"]) + "_" not in f:
      continue
    #print(f)
    if len(results_loop) >= file_limit:
      break
    if f.endswith(".pickle") and cf['dataset']["subset"] in f and f"{short_name}{cf['special_tag']}" + "_"  in f:
      print(f"**********************************reading {f}")
      start_time = time.perf_counter()
      df = pd.read_pickle(os.path.join(drive_path, f))

      df = process_func(df, cf['dataset'], metric_dict, self_conf = SELF_CONF, p_true = PTRUE, thinking = False)
      results_loop.append(df)
      end_time = time.perf_counter()

      elapsed_time = end_time - start_time
      print(f"Execution time: {elapsed_time:.4f} seconds")

  res_loop_df = colab_llm_utils.data_processing.combine_dataframe(results_loop)
  if write_file:
    res_loop_df.to_pickle(os.path.join(drive_path, f"{cf['dataset']['dataset_name']}{short_name}{cf['special_tag']}.pickle"))
  result_dfs.append(res_loop_df)
  print(cf)
  print(f"EM rate: {res_loop_df['em'].mean()}")


**********************************reading routerbench_10k_train_Qwen3-8B_0000.pickle
Execution time: 2.5933 seconds
**********************************reading routerbench_10k_train_Qwen3-8B_0001.pickle
Execution time: 1.5422 seconds
**********************************reading routerbench_10k_train_Qwen3-8B_0002.pickle
Execution time: 2.0085 seconds
**********************************reading routerbench_10k_train_Qwen3-8B_0003.pickle
Execution time: 1.8367 seconds
**********************************reading routerbench_10k_train_Qwen3-8B_0004.pickle
Execution time: 1.4523 seconds
**********************************reading routerbench_10k_train_Qwen3-8B_0005.pickle
Execution time: 1.1272 seconds
**********************************reading routerbench_10k_train_Qwen3-8B_0006.pickle
Execution time: 1.7900 seconds
**********************************reading routerbench_10k_train_Qwen3-8B_0007.pickle
Execution time: 2.1774 seconds
**********************************reading routerbench_10k_train_Qwen3-8B

In [ ]:
res_loop_df.is_correct

,is_correct
0,1
1,1
2,1
3,0
4,1
...,...
3995,1
3996,1
3997,1
3998,1


In [ ]:
for conf, df in zip(merge_configs, result_dfs):
  print(f"CONF: {conf['special_tag']}")
  print(f"EM Rate: {df.em.mean()}")
  print(f"shape: {df.shape}")
  print(f"is_correct: {df.is_correct.mean()}")

CONF: 
EM Rate: 0.70675
shape: (4000, 20)
is_correct: 0.70675
CONF: rag
EM Rate: 0.6695
shape: (4000, 20)
is_correct: 0.6695
CONF: k_shot
EM Rate: 0.74575
shape: (4000, 20)
is_correct: 0.74575
CONF: verify
EM Rate: 0.7015
shape: (4000, 20)
is_correct: 0.7015
CONF: 
EM Rate: 0.79225
shape: (4000, 20)
is_correct: 0.79225


In [ ]:
new_rag[0:1000].em.mean()

In [ ]:
new_rag[1000:2000].em.mean()

In [ ]:
new_rag[2000:3000].em.mean()

In [ ]:
new_rag[3000:4000].em.mean()

In [ ]:
res_13[:1500].em.mean()

In [ ]:
new_rag = result_dfs[0]

In [ ]:
new_rag.em.mean()

In [ ]:
new_rag[new_rag['prompts'] == res_7['prompts']].em.mean()

In [ ]:
res_7[1600:2000][result_dfs[0][1600:2000]['prompts'] != res_7[1600:2000]['prompts']].em.mean()

In [ ]:
result_dfs[0][:200].em.mean()

In [ ]:
o_r_2.prompts.str.contains("Question")

In [ ]:
n_r_2 = result_dfs[1][:100]

In [ ]:
o_r_2 = old_rag[:100]

In [ ]:
[r.get('question', r) for r in n_r_2['source_row']]

In [ ]:
n_r_2[o_r_2['em'] != n_r_2['em']].head(20)

In [ ]:
o_r_2[o_r_2['em'] != n_r_2['em']].head(20)

In [ ]:
new_rag[:100].em.mean()

In [ ]:
vertify = result_dfs[0]

In [ ]:
old_base = res_7[:2000]
new_base = result_dfs[0]
old_rag = res_13[:2000]
new_rag = result_dfs[1][:2000]
larg_mod = result_dfs[-1]

In [ ]:
old_base.shape

In [ ]:
vertify[:1600].em.mean()

In [ ]:
new_base[:1600].em.mean()

In [ ]:
new_rag.head()

In [ ]:
old_rag.em.mean()

In [ ]:
old_rag.is_correct.mean()

In [ ]:
old_rag[ old_rag['is_correct'] == 0 ]

In [ ]:
np.max([len(t) for t in larg_mod.token_outs])

In [ ]:
pd.Series([len(t) for t in old_rag.token_outs]).hist()

In [ ]:
pd.Series([len(t) for t in new_rag.token_outs]).hist()

In [ ]:
old_rag[['prompts']].head(50)

In [ ]:
new_rag[['prompts']].head(50)

In [ ]:
result_dfs[1][['prompts']] == result_dfs[0][['prompts']]

In [ ]:
result_dfs[0].shape

In [ ]:
result_dfs[1][:2000].em.mean()

In [ ]:
result_dfs[1][2000:].em.mean()

In [ ]:
configs.inference.mi

In [ ]:
result_dfs[1][(result_dfs[0]['em'] != result_dfs[1]['em'])].tail()

In [ ]:
for df in result_dfs:
  print(df.is_correct.mean())

In [ ]:
crash

In [ ]:
df.head()

In [ ]:
res_13.em.mean()

In [82]:
cf70 = {"dataset1": colab_llm_utils.configs.datasets.mixeval_ff,
     "dataset2": colab_llm_utils.configs.datasets.mixeval_mc,
   "model": configs.models.llama33_70b_chat,
   "special_tag": "",
        "dict_tag": "tag_70"}
cfk = {"dataset1": colab_llm_utils.configs.datasets.mixeval_ff,
      "dataset2": colab_llm_utils.configs.datasets.mixeval_mc,
    "model": configs.models.qwen3_8b,
    "special_tag": "k_shot",
        "dict_tag": "tag_k_shot"}

cfv = {"dataset1": colab_llm_utils.configs.datasets.mixeval_ff,
      "dataset2": colab_llm_utils.configs.datasets.mixeval_mc,
    "model": configs.models.qwen3_8b,
    "special_tag": "verify",
        "dict_tag": "tag_verify"}

cfr = {"dataset1": colab_llm_utils.configs.datasets.mixeval_ff,
      "dataset2": colab_llm_utils.configs.datasets.mixeval_mc,
    "model": configs.models.qwen3_8b,
    "special_tag": "rag",
        "dict_tag": "tag_rag"}

cf1 = {"dataset1": colab_llm_utils.configs.datasets.mixeval_ff,
      "dataset2": colab_llm_utils.configs.datasets.mixeval_mc,
    "model": configs.models.qwen3_8b,
    "special_tag": "",
        "dict_tag": "tag_"}


METRIC = "gpt_score"
METRIC_2 = "em"
mixeval_cfgs= [cf70, cfk, cfv, cfr, cf1]

In [17]:
cf70 = {"dataset1": colab_llm_utils.configs.datasets.routerbench_10k,
     "dataset2": colab_llm_utils.configs.datasets.routerbench_rag,
   "model": configs.models.llama33_70b_chat,
   "special_tag": ""}
cfk = {"dataset1": colab_llm_utils.configs.datasets.routerbench_10k,
      "dataset2": colab_llm_utils.configs.datasets.routerbench_rag,
    "model": configs.models.qwen3_8b,
    "special_tag": "k_shot"}

cfv = {"dataset1": colab_llm_utils.configs.datasets.routerbench_10k,
      "dataset2": colab_llm_utils.configs.datasets.routerbench_rag,
    "model": configs.models.qwen3_8b,
    "special_tag": "verify"}

cfr = {"dataset1": colab_llm_utils.configs.datasets.routerbench_10k,
      "dataset2": colab_llm_utils.configs.datasets.routerbench_rag,
    "model": configs.models.qwen3_8b,
    "special_tag": "rag"}

cf1 = {"dataset1": colab_llm_utils.configs.datasets.routerbench_10k,
      "dataset2": colab_llm_utils.configs.datasets.routerbench_rag,
    "model": configs.models.qwen3_8b,
    "special_tag": ""}

METRIC = "is_correct"
METRIC_2 = "gpt_score"


In [23]:
dfs = {}


In [40]:

cf=cf1

SAVE_FILE =False
STACK_MIX = True
REPLACE = False
if STACK_MIX:
  short_name = cf['model']['model_name'].split('/')[-1]
  special_tag = cf['special_tag']
  dataset = cf['dataset1']
  dataset2 = cf['dataset2']
  df_res1 = pd.read_pickle(os.path.join(f"/content/drive/MyDrive/phase3/Llama/{dataset['clean_name']}/", f"{dataset['dataset_name']}{short_name}{special_tag}.pickle"))
  df_res2 = pd.read_pickle(os.path.join(f"/content/drive/MyDrive/phase3/Llama/{dataset2['clean_name']}/", f"{dataset2['dataset_name']}{short_name}{special_tag}.pickle"))
  if REPLACE:
    df_res1 = df_res1[:-len(df_res2)]
  df_res2[METRIC] = (df_res2[METRIC_2] > .1).astype(int)
  combined = pd.concat([df_res1, df_res2])
  dfs["tag_" + special_tag] = combined
  print(f"Special tag {special_tag}")
  print(f"mean score: {combined[:len(df_res1)][METRIC].mean()}")
  print(f"mean score: {combined[len(df_res1):][METRIC].mean()}")
  print(f"final shape {combined.shape}")
  if SAVE_FILE:
    pd.concat([df_res1, df_res2]).to_pickle(os.path.join(f"/content/drive/MyDrive/phase3/Llama/{dataset['clean_name']}/", f"{dataset['dataset_name']}{short_name}{special_tag}_stack.pickle"))

Special tag 
mean score: 0.5145
mean score: 0.7765
final shape (4000, 21)


In [45]:
report_met = "f1"
for k in dfs.keys():
  print(k)
  print(dfs[k].shape)
  print((dfs[k][:2000][report_met] > .5).mean())

tag_
(4000, 21)
0.318
tag_k_shot
(4000, 23)
0.526
tag_verify
(4000, 24)
0.434
tag_rag
(4000, 23)
0.497
tag_70
(4000, 23)
0.695


In [31]:
combined.head(20)

,prompts,responses,ans,logit_outs,token_outs,blocks,meta,task_family,source_row,likes,...,chow_sum,chow_quantile,log_chow_av,thinking_text,scored_responses,rouge,em,f1,gpt_score,is_correct
0,Answer the following question with only the co...,[James Ellroy],[JAMES ELROY],"[{29184: 37.25, 13379: 22.125, 43810: 20.625, ...","[128000, 128000, 128006, 9125, 128007, 271, 38...",{},{},unknown,{},"[37.25, 39.5, 41.5, 25.9375]",...,3.998927,1.000000,-0.000268,None,[James Ellroy],0.500000,0.0,0.500000,1,NaN
1,Answer the following question with only the co...,[1 year],"[1, 1]","[{16: 36.45833206176758, 4054: 26.375, 17: 23....","[128000, 128000, 128006, 9125, 128007, 271, 38...",{},{},unknown,{},"[36.45833206176758, 28.375, 24.583332061767578]",...,2.981612,0.992039,-0.006158,None,[1 year],0.666667,0.0,0.666667,1,NaN
2,Answer the following question with only the co...,[Vitamin B1 (Thiamine)],"[B 1, B-1, B-1 (disambiguation), B.1, B1 (disa...","[{53: 40.75, 1016: 33.75, 33: 25.875, 45774: 2...","[128000, 128000, 128006, 9125, 128007, 271, 38...",{},{},unknown,{},"[40.75, 40.0, 41.75, 34.0, 31.0, 41.0, 37.75, ...",...,9.997133,0.999971,-0.000287,None,[Vitamin B1 (Thiamine)],0.500000,0.0,0.500000,1,NaN
3,Answer the following question with only the co...,[Aleister Crowley],"[Aleister Crowly, Alesteir Crowley, Alistair c...","[{84583: 39.0, 19623: 28.25, 32: 24.5, 25940: ...","[128000, 128000, 128006, 9125, 128007, 271, 38...",{},{},unknown,{},"[39.0, 38.25, 38.5, 26.458332061767578]",...,3.999553,0.999989,-0.000112,None,[Aleister Crowley],0.500000,0.0,0.500000,1,NaN
4,Answer the following question with only the co...,[Elephant],"[Elephantinely, Elephant tusks, Rogue Elephant...","[{46439: 35.25, 791: 28.625, 79189: 27.375, 10...","[128000, 128000, 128006, 9125, 128007, 271, 38...",{},{},unknown,{},"[35.25, 41.5, 24.6875]",...,2.997804,0.999590,-0.000732,None,[Elephant],1.000000,1.0,1.000000,1,NaN
5,Answer the following question with only the co...,[Gastrointestinal tract],"[Gastro-esophageal junction, Anatomy of the hu...","[{38: 34.25, 626: 30.75, 267: 29.75, 70: 29.25...","[128000, 128000, 128006, 9125, 128007, 271, 38...",{},{},unknown,{},"[34.25, 37.5, 32.5, 24.625, 22.5]",...,4.691024,0.996912,-0.070233,None,[Gastrointestinal tract],0.000000,0.0,0.000000,0,NaN
6,Answer the following question with only the co...,[Southwest Airlines],"[RedBellyRadio, Southwest Airlines Inc, Blogso...","[{26070: 33.25, 46785: 23.625, 17268: 21.375, ...","[128000, 128000, 128006, 9125, 128007, 271, 38...",{},{},unknown,{},"[33.25, 39.5, 32.25, 24.79166603088379]",...,3.998522,0.999946,-0.000370,None,[Southwest Airlines],1.000000,1.0,1.000000,1,NaN
7,Answer the following question with only the co...,[Painkiller],"[Pain killing, Pain killer, Analgesic drugs, A...","[{47: 34.5, 92724: 32.5, 857: 19.875, 35567: 1...","[128000, 128000, 128006, 9125, 128007, 271, 38...",{},{},unknown,{},"[34.5, 33.75, 32.75, 25.208332061767578]",...,3.779833,0.949598,-0.058171,None,[Painkiller],0.000000,0.0,0.000000,1,NaN
8,Answer the following question with only the co...,[Vitamin D],"[Calcium supplements, Calcium ion, Factor 4, C...","[{53: 39.0, 48268: 30.875, 27684: 26.0, 55911:...","[128000, 128000, 128006, 9125, 128007, 271, 38...",{},{},unknown,{},"[39.0, 41.5, 38.25, 21.875]",...,3.866070,0.999850,-0.035936,None,[Vitamin D],0.000000,0.0,0.000000,0,NaN
9,Answer the following question with only the co...,[9 years old for night work and some other rol...,[Nine years],"[{24: 41.5, 23: 28.25, 84429: 27.25, 806: 24.7...","[128000, 128000, 128006, 9125, 128007, 271, 38...",{},{},unknown,{},"[41.5, 22.5, 30.125, 22.75, 23.5, 23.958332061...",...,41.596047,0.678734,-0.431654,None,[9 years old for night work and some other rol...,0.035714,0.0,0.037736,0,NaN


In [63]:
  ind_tag = "tag_rag"
  dfs[ind_tag][dfs[ind_tag]['em'] > dfs[ind_tag]['gpt_score']]

,prompts,responses,ans,logit_outs,token_outs,blocks,meta,task_family,source_row,likes,...,chow_sum,chow_quantile,log_chow_av,thinking_text,scored_responses,rouge,em,f1,gpt_score,is_correct
35,Answer the following question with only the co...,[Vientiane],"[Vientiane, Luang Prabang]","[{53: 63.75, 49202: 51.75, 43: 40.25, 94054: 3...","[151644, 872, 198, 16141, 279, 2701, 3405, 448...",{},{},unknown,{},"[63.75, 34.791664123535156, 35.83333206176758,...",...,3.999498,0.999952,-0.000126,None,[Vientiane],1.0,1.0,1.0,0,NaN
1216,Supporting context sources may or may not cont...,[Circle],[Centre for Innovation Research and Competence...,"[{25199: 46.5, 51942: 37.0, 25857: 34.0, 32: 3...","[151644, 872, 198, 7916, 287, 2266, 8173, 1231...",{},{},unknown,{},"[46.5, 28.958332061767578]",...,1.999662,0.999831,-0.000169,None,[Circle],1.0,1.0,1.0,0,NaN
1747,Supporting context sources may or may not cont...,[Mrs. White],"[List of Cluedo characters, Miss Scarlett, Mrs...","[{49229: 54.5, 12275: 50.25, 35312: 49.25, 470...","[151644, 872, 198, 7916, 287, 2266, 8173, 1231...",{},{},unknown,{},"[54.5, 49.375, 45.416664123535156, 29.79166603...",...,3.977888,0.998513,-0.005575,None,[Mrs. White],1.0,1.0,1.0,0,NaN


In [72]:
dfs["tag_verify"]

,prompts,responses,ans,logit_outs,token_outs,blocks,meta,task_family,source_row,likes,...,chow_quantile,log_chow_av,thinking_text,scored_responses,rouge,em,f1,verify_prompt,gpt_score,is_correct
0,QUESTION:\nAnswer the following question with ...,[James Ellroy],[JAMES ELROY],"[{28084: 27.25, 16: 20.0, 32: 19.875, 59662: 1...","[151644, 872, 198, 19432, 279, 9144, 4226, 368...",{},{},unknown,{},"[27.25, 26.0, 25.375, 29.583332061767578]",...,0.997667,-1.378373e-01,None,[James Ellroy],0.500000,0.0,0.5,Review the candidate answer below.\n\nQUESTION...,1,NaN
1,QUESTION:\nAnswer the following question with ...,[1],"[1, 1]","[{16: 46.875, 17: 39.166664123535156, 18: 30.6...","[151644, 872, 198, 19432, 279, 9144, 4226, 368...",{},{},unknown,{},"[46.875, 30.208332061767578]",...,0.999775,-2.255142e-04,None,[1],1.000000,1.0,1.0,Review the candidate answer below.\n\nQUESTION...,1,NaN
2,QUESTION:\nAnswer the following question with ...,[Thiamine],"[B 1, B-1, B-1 (disambiguation), B.1, B1 (disa...","[{1001: 42.0, 33: 34.25, 53: 32.25, 339: 29.25...","[151644, 872, 198, 19432, 279, 9144, 4226, 368...",{},{},unknown,{},"[42.0, 41.5, 48.5, 28.125]",...,0.998789,-2.315064e-03,None,[Thiamine],0.000000,0.0,0.0,Review the candidate answer below.\n\nQUESTION...,1,NaN
3,QUESTION:\nAnswer the following question with ...,[Aleister Crowley],"[Aleister Crowly, Alesteir Crowley, Alistair c...","[{83483: 34.5, 66622: 25.375, 2101: 25.0, 32: ...","[151644, 872, 198, 19432, 279, 9144, 4226, 368...",{},{},unknown,{},"[34.5, 35.75, 34.75, 30.208332061767578]",...,0.999978,-8.970428e-05,None,[Aleister Crowley],0.500000,0.0,0.5,Review the candidate answer below.\n\nQUESTION...,1,NaN
4,QUESTION:\nAnswer the following question with ...,[The elephant],"[Elephantinely, Elephant tusks, Rogue Elephant...","[{785: 43.75, 45339: 40.0, 10068: 27.875, 1002...","[151644, 872, 198, 19432, 279, 9144, 4226, 368...",{},{},unknown,{},"[43.75, 44.791664123535156, 30.833332061767578]",...,0.999995,-7.750733e-03,None,[The elephant],0.666667,1.0,1.0,Review the candidate answer below.\n\nQUESTION...,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,Review the candidate answer below.\n\nQUESTION...,[A],[A],"[{32: 48.541664123535156, 34: 27.9166660308837...","[151644, 872, 198, 19432, 279, 9144, 4226, 368...",{},{},unknown,{},"[48.541664123535156, 30.41666603088379]",...,0.999994,-6.133693e-06,None,[A],NaN,1.0,NaN,NaN,1,1.0
1996,Review the candidate answer below.\n\nQUESTION...,[A],[A],"[{32: 55.416664123535156, 34: 36.875, 33: 34.3...","[151644, 872, 198, 19432, 279, 9144, 4226, 368...",{},{},unknown,{},"[55.416664123535156, 30.208332061767578]",...,0.999995,-4.856190e-06,None,[A],NaN,1.0,NaN,NaN,1,1.0
1997,Review the candidate answer below.\n\nQUESTION...,[B],[B],"[{33: 48.541664123535156, 34: 31.0416660308837...","[151644, 872, 198, 19432, 279, 9144, 4226, 368...",{},{},unknown,{},"[48.541664123535156, 30.208332061767578]",...,0.999999,-1.040513e-06,None,[B],NaN,1.0,NaN,NaN,1,1.0
1998,Review the candidate answer below.\n\nQUESTION...,[B],[B],"[{33: 46.666664123535156, 34: 32.5, 32: 28.125...","[151644, 872, 198, 19432, 279, 9144, 4226, 368...",{},{},unknown,{},"[46.666664123535156, 30.625]",...,0.999999,-1.480621e-06,None,[B],NaN,1.0,NaN,NaN,1,1.0


In [70]:
dfs.keys()

dict_keys(['tag_', 'tag_k_shot', 'tag_verify', 'tag_rag', 'tag_70'])

In [74]:
for tag, df in dfs.items():
    dfs[tag] = df.reset_index(drop=True)

In [75]:
def make_hashable(value):
    if isinstance(value, (list, tuple)):
        return tuple(make_hashable(v) for v in value)
    if isinstance(value, dict):
        return tuple(sorted((k, make_hashable(v)) for k, v in value.items()))
    return value

combined = pd.concat(
    [
        df.assign(source=tag, question_id=df.index)
        for tag, df in dfs.items()
    ],
    ignore_index=True,
)

combined["_response_key"] = combined["scored_responses"].map(make_hashable)

mask = (
    combined.groupby(
        ["question_id", "_response_key"], dropna=False
    )["gpt_score"]
    .transform("nunique")
    .gt(1)
)

discrepancies = (
    combined.loc[mask]
    .sort_values(["question_id", "source"])
    .drop(columns="_response_key")
)

display(discrepancies[
    ["question_id", "source", "scored_responses", "gpt_score", "f1", "ans", "prompts"]
])

,question_id,source,scored_responses,gpt_score,f1,ans,prompts
16010,10,tag_70,[Google Buzz],1,0.000000,"[Waving, Speed of a wave, Wave motion in liqui...",Answer the following question with only the co...
12010,10,tag_rag,[Google Buzz],0,0.000000,"[Waving, Speed of a wave, Wave motion in liqui...",Answer the following question with only the co...
60,60,tag_,[ In the beginning God created\n\n],1,0.857143,[IN THE BEGINNING GOD],Answer the following question with only the co...
4060,60,tag_k_shot,[ In the beginning God created\n\n],0,0.857143,[IN THE BEGINNING GOD],Answer the following question with only the co...
71,71,tag_,[ A\n\n],0,0.000000,[(B)],Answer the following question with only the co...
4071,71,tag_k_shot,[ A\n\n],1,0.000000,[(B)],Answer the following question with only the co...
16230,230,tag_70,[Google Buzz],1,0.000000,"[Waving, Speed of a wave, Wave motion in liqui...",Answer the following question with only the co...
12230,230,tag_rag,[Google Buzz],0,0.000000,"[Waving, Speed of a wave, Wave motion in liqui...",Answer the following question with only the co...
16274,274,tag_70,[Google Buzz],1,0.000000,"[Waving, Speed of a wave, Wave motion in liqui...",Answer the following question with only the co...
12274,274,tag_rag,[Google Buzz],0,0.000000,"[Waving, Speed of a wave, Wave motion in liqui...",Answer the following question with only the co...


In [91]:
for df in dfs.values():
  print(df.em.mean() )
  print(df.em.isna().sum() )

0.5175
0
0.59175
0
0.55575
0
0.5685
0
0.68475
0


In [80]:
def update_scores_from_other_dfs(dfs, target_tag):
    target = dfs[target_tag]
    corrected = target["em"].eq(1).copy()

    target_keys = target["scored_responses"].map(make_hashable)

    for tag, other in dfs.items():
        if tag == target_tag:
            continue

        other_keys = (
            other["scored_responses"]
            .map(make_hashable)
            .reindex(target.index)
        )
        other_scores = other["gpt_score"].reindex(target.index)

        corrected |= target_keys.eq(other_keys) & other_scores.eq(1)

    changed = corrected & target["gpt_score"].ne(1)
    target.loc[changed, "gpt_score"] = 1

    return int(changed.sum())


n_updated = update_scores_from_other_dfs(dfs, target_tag="tag_k_shot")
print(f"Updated {n_updated} scores")

Updated 1 scores


In [84]:
SAVE_FILE = True
for cf in mixeval_cfgs:
  short_name = cf['model']['model_name'].split('/')[-1]
  special_tag = cf['special_tag']
  dataset = cf['dataset1']
  dataset2 = cf['dataset2']
  if SAVE_FILE:
    dataset = cf['dataset1']
    dfs[cf['dict_tag']].to_pickle(os.path.join(f"/content/drive/MyDrive/phase3/Llama/{dataset['clean_name']}/", f"{dataset['dataset_name']}{short_name}{special_tag}_stack.pickle"))


In [85]:
  dfs["tag_70"][int_cols].head(50)

,prompts,responses,ans,em,gpt_score
0,Answer the following question with only the co...,[James Ellroy],[JAMES ELROY],0.0,1
1,Answer the following question with only the co...,[1 year],"[1, 1]",0.0,1
2,Answer the following question with only the co...,[Vitamin B1 (Thiamine)],"[B 1, B-1, B-1 (disambiguation), B.1, B1 (disa...",0.0,1
3,Answer the following question with only the co...,[Aleister Crowley],"[Aleister Crowly, Alesteir Crowley, Alistair c...",0.0,1
4,Answer the following question with only the co...,[Elephant],"[Elephantinely, Elephant tusks, Rogue Elephant...",1.0,1
5,Answer the following question with only the co...,[Gastrointestinal tract],"[Gastro-esophageal junction, Anatomy of the hu...",0.0,0
6,Answer the following question with only the co...,[Southwest Airlines],"[RedBellyRadio, Southwest Airlines Inc, Blogso...",1.0,1
7,Answer the following question with only the co...,[Painkiller],"[Pain killing, Pain killer, Analgesic drugs, A...",0.0,1
8,Answer the following question with only the co...,[Vitamin D],"[Calcium supplements, Calcium ion, Factor 4, C...",0.0,0
9,Answer the following question with only the co...,[9 years old for night work and some other rol...,[Nine years],0.0,0


In [53]:
  int_cols = ['prompts', 'responses', 'ans', 'em','gpt_score']
  dfs["tag_"].head(20)

,prompts,responses,ans,logit_outs,token_outs,blocks,meta,likes,all_probas,top_probas,...,chow_sum,chow_quantile,log_chow_av,thinking_text,scored_responses,rouge,em,f1,gpt_score,is_correct
0,Answer the following question with only the co...,[1\n],[JAMES ELROY],"[{16: 24.375, 17: 20.75, 15: 20.375, 19: 19.75...","[16141, 279, 2701, 3405, 448, 1172, 279, 4396,...",{},{},"[24.375, 24.0]","[[0.9232263113346861, 0.024603147834242483, 0....","[0.9232263113346861, 0.35569104999617207]",...,1.278917,0.639459,-0.556787,None,[1\n],0.000000,0.0,0.000000,1,NaN
1,Answer the following question with only the co...,[1\n],"[1, 1]","[{16: 30.208332061767578, 17: 25.625, 15: 22.9...","[16141, 279, 2701, 3405, 448, 1172, 279, 4396,...",{},{},"[30.208332061767578, 31.375]","[[0.98823785579254, 0.010100564887465188, 0.00...","[0.98823785579254, 0.5986103495967917]",...,1.586848,0.793424,-0.262488,None,[1\n],1.000000,1.0,1.000000,1,NaN
2,Answer the following question with only the co...,[1\n],"[B 1, B-1, B-1 (disambiguation), B.1, B1 (disa...","[{16: 23.25, 425: 22.25, 44674: 21.75, 663: 21...","[16141, 279, 2701, 3405, 448, 1172, 279, 4396,...",{},{},"[23.25, 27.625]","[[0.5036612539174818, 0.18528662063087115, 0.1...","[0.5036612539174818, 0.5167431125338025]",...,1.020404,0.510202,-0.673030,None,[1\n],0.666667,0.0,0.666667,1,NaN
3,Answer the following question with only the co...,[1\n],"[Aleister Crowly, Alesteir Crowley, Alistair c...","[{16: 22.0, 15: 21.875, 19081: 21.125, 17: 19....","[16141, 279, 2701, 3405, 448, 1172, 279, 4396,...",{},{},"[22.0, 24.0]","[[0.39209620476797846, 0.34602368622291624, 0....","[0.39209620476797846, 0.45627490157241773]",...,0.848371,0.424186,-0.860454,None,[1\n],0.000000,0.0,0.000000,1,NaN
4,Answer the following question with only the co...,[ The elephant\n],"[Elephantinely, Elephant tusks, Rogue Elephant...","[{576: 22.0, 16: 21.375, 15: 19.875, 19: 19.12...","[16141, 279, 2701, 3405, 448, 1172, 279, 4396,...",{},{},"[22.0, 23.75, 25.5]","[[0.5338271455229991, 0.2857370804948554, 0.06...","[0.5338271455229991, 0.6214729141051121, 0.486...",...,1.641359,0.533827,-0.608257,None,[ The elephant\n],0.666667,1.0,1.000000,1,NaN
5,Answer the following question with only the co...,[ Stomach\n],"[Gastro-esophageal junction, Anatomy of the hu...","[{794: 24.25, 22350: 24.0, 16: 22.875, 9600: 2...","[16141, 279, 2701, 3405, 448, 1172, 279, 4396,...",{},{},"[24.25, 38.0, 39.75, 29.125]","[[0.4327344800500371, 0.33701395192496614, 0.1...","[0.4327344800500371, 0.99999930521189, 0.99999...",...,3.203210,0.885238,-0.274595,None,[ Stomach\n],0.500000,0.0,0.500000,1,NaN
6,Answer the following question with only the co...,[1\n\n],"[RedBellyRadio, Southwest Airlines Inc, Blogso...","[{45685: 23.125, 16: 23.125, 7568: 19.875, 168...","[16141, 279, 2701, 3405, 448, 1172, 279, 4396,...",{},{},"[23.125, 22.875]","[[0.44949858339636184, 0.44949858339636184, 0....","[0.44949858339636184, 0.5038506004114324]",...,0.953349,0.476675,-0.742549,None,[1\n\n],0.000000,0.0,0.000000,0,NaN
7,Answer the following question with only the co...,[ A painkiller\n],"[Pain killing, Pain killer, Analgesic drugs, A...","[{362: 22.5, 16: 22.0, 6646: 21.625, 34467: 21...","[16141, 279, 2701, 3405, 448, 1172, 279, 4396,...",{},{},"[22.5, 30.125, 37.25, 29.375]","[[0.37353195896916735, 0.2265585854973214, 0.1...","[0.37353195896916735, 0.9922756003508926, 0.67...",...,2.888408,0.761300,-0.387599,None,[ A painkiller\n],0.000000,0.0,0.000000,1,NaN
8,Answer the following question with only the co...,[1\n],"[Calcium supplements, Calcium ion, Factor 4, C...","[{16: 23.0, 44674: 22.5, 17: 22.0, 18: 20.875,...","[16141, 279, 2701, 3405, 448, 1172, 279, 4396,...",{},{},"[23.0, 27.375]","[[0.42329570764462243, 0.2567418248112189, 0.1...","[0.42329570764462243, 0.5184261880309191]",...,0.941722,0.470861,-0.758321,None,[1\n],0.000000,0.0,0.000000,1,NaN
9,Answer the following question with only the co...,[9 years old\n],[Nine years],"[{24: 31.0, 23: 23.85416603088379, 16: 

In [ ]:
combined[:2000]

In [ ]:
# READ STACKED
cf = cfv
short_name = cf['model']['model_name'].split('/')[-1]
dataset = cf['dataset1']
special_tag = cf['special_tag']
pd.read_pickle(os.path.join(f"/content/drive/MyDrive/phase3/Llama/{dataset['clean_name']}/", f"{dataset['dataset_name']}{short_name}{special_tag}_stack.pickle"))

In [ ]:
combined.head()

In [ ]:
df_res2.head()

In [ ]:
combined.gpt_score.mean()

In [ ]:
combined.head()

In [ ]:
res_7[res_7['task_family'] =='arc-challenge'][['prompts', 'responses', 'ans', 'is_correct', 'responses_32b', '32b_is_correct']]

In [ ]:
for idx, row in res_13[res_13['task_family'] =='mmlu'].iterrows():
  correct_ans = colab_llm_utils.data_processing.find_mmlu_correct_answer(row)
  matching = correct_ans == row['ans']
  if not matching:
    print(f"original ans: {row['ans']}")
    print(f"modified ans: {correct_ans}")


In [ ]:
res_13[res_13['task_family'] =='mmlu'][['prompts','source_row']].head(10)

In [ ]:
res_13[res_13['task_family'] =='mmlu'][['prompts', 'responses', 'ans', 'is_correct']].head(50)

In [ ]:
res_13[res_13['prompts'].str.contains("Score: 0.02")]

In [ ]:
#print(res_13.loc[71, :])
res_13.loc[[79]][['prompts', 'responses', 'is_correct', 'source_row']]
#print(res_13.loc[79, :])

In [ ]:
res_13[res_13['task_family'] =='mmlu'][['prompts', 'responses', 'ans', 'is_correct', 'source_row']].head(25)

In [ ]:
def audit_answers_by_task(df):
    """
    Generates a crosstab of gold answers ('ans') against 'task_family'
    to verify answer formatting and index alignment.
    """
    # Crosstab count of each answer type per task family
    ct = pd.crosstab(
        df['task_family'],
        df['ans'].astype(str),
        margins=True,
        margins_name="TOTAL"
    )

    print("=== Target Answer ('ans') Crosstab by Task Family ===")
    display(ct)

    # Check for unexpected numeric answers in MCQ task families
    mcq_families = ['mmlu', 'arc-challenge', 'hellaswag', 'winogrande']
    mcq_mask = df['task_family'].isin(mcq_families)
    numeric_ans_count = df[mcq_mask]['ans'].astype(str).str.isdigit().sum()

    if numeric_ans_count > 0:
        print(f"\n⚠️ WARNING: Found {numeric_ans_count} unmapped numeric answer targets in MCQ families!")
        print("Run the updated 0-indexed doc_to_ans logic to normalize these to uppercase letters.")
    else:
        print("\n✅ All MCQ answer targets are properly mapped to letter options.")

    return ct

# --- Run on your current dataframe ---
mcq_families = ['mmlu', 'arc-challenge', 'hellaswag', 'winogrande']
mcq_mask = res_7[res_7['task_family'].isin(mcq_families)]
ct_summary = audit_answers_by_task(mcq_mask)